# Lab : Image Classification using Convolutional Neural Networks

At the end of this laboratory, you would get familiarized with

*   Creating deep networks using Keras
*   Steps necessary in training a neural network
*   Prediction and performance analysis using neural networks

---

# **In case you use a colaboratory environment**
By default, Colab notebooks run on CPU.
You can switch your notebook to run with GPU.

In order to obtain access to the GPU, you need to choose the tab Runtime and then select “Change runtime type” as shown in the following figure:

![Changing runtime](https://miro.medium.com/max/747/1*euE7nGZ0uJQcgvkpgvkoQg.png)

When a pop-up window appears select GPU. Ensure “Hardware accelerator” is set to GPU.

# **Working with a new dataset: CIFAR-10**

The CIFAR-10 dataset consists of 60000 32x32 colour images in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images. More information about CIFAR-10 can be found [here](https://www.cs.toronto.edu/~kriz/cifar.html).

In Keras, the CIFAR-10 dataset is also preloaded in the form of four Numpy arrays. x_train and y_train contain the training set, while x_test and y_test contain the test data. The images are encoded as Numpy arrays and their corresponding labels ranging from 0 to 9.

Your task is to:

*   Visualize the images in CIFAR-10 dataset. Create a 10 x 10 plot showing 10 random samples from each class.
*   Convert the labels to one-hot encoded form.
*   Normalize the images.




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

keras.utils.set_random_seed(42)
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

In [ ]:
class_names = np.array([
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
])
rng = np.random.default_rng(42)
fig, axes = plt.subplots(10, 10, figsize=(12, 12))
for label in range(10):
    indices = rng.choice(np.flatnonzero(y_train.ravel() == label), 10, replace=False)
    for column, index in enumerate(indices):
        axes[label, column].imshow(x_train[index])
        axes[label, column].set_xticks([])
        axes[label, column].set_yticks([])
    axes[label, 0].set_ylabel(class_names[label])
plt.suptitle('Ten random CIFAR-10 images per class')
plt.tight_layout()
plt.show()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_test_labels = y_test.ravel().copy()
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Fixed, shuffled validation holdout shared by all three models.
indices = rng.permutation(len(x_train))
val_indices, train_indices = indices[:5000], indices[5000:]
x_val, y_val = x_train[val_indices], y_train[val_indices]
x_fit, y_fit = x_train[train_indices], y_train[train_indices]
print('Training:', x_fit.shape, 'Validation:', x_val.shape, 'Test:', x_test.shape)

## Define the following model (same as the one in tutorial)

For the convolutional front-end, start with a single convolutional layer with a small filter size (3,3) and a modest number of filters (32) followed by a max pooling layer. 

Use the input as (32,32,3). 

The filter maps can then be flattened to provide features to the classifier. 

Use a dense layer with 100 units before the classification layer (which is also a dense layer with softmax activation).

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

In [ ]:
baseline = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(100, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='baseline_cnn')
baseline.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 50 epochs with a batch size of 512.

In [ ]:
baseline.compile(optimizer=keras.optimizers.SGD(),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_baseline = baseline.fit(
    x_fit, y_fit, validation_data=(x_val, y_val), epochs=50, batch_size=512
)

*   Plot the cross entropy loss curve and the accuracy curve

In [ ]:
def plot_histories(histories):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, history in histories.items():
        epochs = np.arange(1, len(history.history['loss']) + 1)
        for ax, metric, title in zip(axes, ['loss', 'accuracy'],
                                     ['Categorical crossentropy', 'Accuracy']):
            line, = ax.plot(epochs, history.history[metric], label=f'{name}: train')
            ax.plot(epochs, history.history[f'val_{metric}'], '--',
                    color=line.get_color(), label=f'{name}: validation')
            ax.set(xlabel='Epoch', ylabel=title, title=title)
            ax.legend()
            ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

plot_histories({'Baseline': history_baseline})

## Defining Deeper Architectures: VGG Models

*   Define a deeper model architecture for CIFAR-10 dataset and train the new model for 50 epochs with a batch size of 512. We will use VGG model as the architecture.

Stack two convolutional layers with 32 filters, each of 3 x 3. 

Use a max pooling layer and next flatten the output of the previous layer and add a dense layer with 128 units before the classification layer. 

For all the layers, use ReLU activation function. 

Use same padding for the layers to ensure that the height and width of each layer output matches the input


In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

In [ ]:
deeper = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='two_convolution_vgg')
deeper.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 50 epochs with a batch size of 512.

In [ ]:
deeper.compile(optimizer=keras.optimizers.SGD(),
               loss='categorical_crossentropy', metrics=['accuracy'])
history_deeper = deeper.fit(
    x_fit, y_fit, validation_data=(x_val, y_val), epochs=50, batch_size=512
)

*   Compare the performance of both the models by plotting the loss and accuracy curves of both the training steps. Does the deeper model perform better? Comment on the observation.
 

In [ ]:
plot_histories({'Baseline': history_baseline, 'Deeper': history_deeper})
base_val = history_baseline.history['val_accuracy'][-1]
deep_val = history_deeper.history['val_accuracy'][-1]
print(f'Final validation accuracy: baseline={base_val:.2%}, deeper={deep_val:.2%}')
print(f'Deeper model difference: {(deep_val - base_val) * 100:+.2f} percentage points')
if deep_val > base_val:
    print('The deeper model performs better on final validation accuracy.')
else:
    print('The deeper model does not outperform the baseline on final validation accuracy.')
for name, h in [('Baseline', history_baseline), ('Deeper', history_deeper)]:
    gap = h.history['accuracy'][-1] - h.history['val_accuracy'][-1]
    print(f'{name} final train-minus-validation accuracy gap: {gap:.2%}')

**Performance interpretation**

The preceding cell reports the measured comparison after training. Higher validation accuracy and lower validation crossentropy indicate better generalization. Extra convolutional depth can learn richer features, but improvement is not guaranteed. A widening training–validation accuracy gap, especially with rising validation loss while training loss falls, suggests overfitting. Compare validation curves rather than training accuracy alone.

No numerical results are claimed before execution. Both models use the same validation split, optimizer, batch size and epoch count; their different padding and dense layers mean this is not an isolated test of depth alone.

*   Use predict function to predict the output for the test split
*   Plot the confusion matrix for the new model and comment on the class confusions.


In [ ]:
def show_confusions(true_labels, predicted_labels, title):
    cm = tf.math.confusion_matrix(true_labels, predicted_labels, num_classes=10).numpy()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, cmap='Blues')
    fig.colorbar(im, ax=ax)
    ax.set(xticks=np.arange(10), yticks=np.arange(10),
           xticklabels=class_names, yticklabels=class_names,
           xlabel='Predicted class', ylabel='True class', title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    for i in range(10):
        for j in range(10):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    plt.tight_layout()
    plt.show()
    errors = cm.copy()
    np.fill_diagonal(errors, 0)
    print('Largest observed class confusions (true → predicted):')
    for index in np.argsort(errors.ravel())[::-1][:5]:
        i, j = np.unravel_index(index, errors.shape)
        if errors[i, j] > 0:
            print(f'{class_names[i]} → {class_names[j]}: {errors[i, j]} images '
                  f'({errors[i, j] / cm[i].sum():.1%} of true {class_names[i]})')
    return cm

predictions_deeper = deeper.predict(x_test, batch_size=512)
y_pred_deeper = predictions_deeper.argmax(axis=1)
cm_deeper = show_confusions(y_test_labels, y_pred_deeper, 'Two-convolution VGG: test set')

**Class confusions**

Rows represent true classes and columns represent predictions. Diagonal counts are correct classifications; off-diagonal counts are errors. The preceding cell identifies the five largest observed confusions and their proportions within the true class. Similar appearance and low image resolution can explain errors between related classes, such as cats and dogs or automobiles and trucks, but these are possible explanations rather than measured findings until the notebook runs.

*    Print the test accuracy for the trained model.

In [ ]:
test_loss_deeper, test_accuracy_deeper = deeper.evaluate(x_test, y_test, batch_size=512, verbose=0)
print(f'Two-convolution VGG test loss: {test_loss_deeper:.4f}')
print(f'Two-convolution VGG test accuracy: {test_accuracy_deeper:.2%}')

## Define the complete VGG architecture.

Stack two convolutional layers with 64 filters, each of 3 x 3 followed by max pooling layer. 

Stack two more convolutional layers with 128 filters, each of 3 x 3, followed by max pooling, followed by two more convolutional layers with 256 filters, each of 3 x 3, followed by max pooling. 

Flatten the output of the previous layer and add a dense layer with 128 units before the classification layer. 

For all the layers, use ReLU activation function. 

Use same padding for the layers to ensure that the height and width of each layer output matches the input

*   Change the size of input to 64 x 64.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(42)

In [ ]:
# Resize per batch to avoid storing all 64×64 images in memory.
def make_dataset(images, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        dataset = dataset.shuffle(5000, seed=42, reshuffle_each_iteration=True)
    dataset = dataset.batch(512)
    dataset = dataset.map(
        lambda x, y: (tf.image.resize(x, (64, 64)), y),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return dataset.prefetch(1)

train64 = make_dataset(x_fit, y_fit, training=True)
val64 = make_dataset(x_val, y_val)
test64 = make_dataset(x_test, y_test)

full_vgg = keras.Sequential([
    keras.Input(shape=(64, 64, 3)),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='full_vgg')
full_vgg.summary()

*   Compile the model using categorical_crossentropy loss, SGD optimizer and use 'accuracy' as the metric.
*   Use the above defined model to train CIFAR-10 and train the model for 10 epochs with a batch size of 512.
*   Predict the output for the test split and plot the confusion matrix for the new model and comment on the class confusions.

In [ ]:
full_vgg.compile(optimizer=keras.optimizers.SGD(),
                 loss='categorical_crossentropy', metrics=['accuracy'])
history_full = full_vgg.fit(train64, validation_data=val64, epochs=10)
plot_histories({'Full VGG': history_full})
predictions_full = full_vgg.predict(test64)
y_pred_full = predictions_full.argmax(axis=1)
cm_full = show_confusions(y_test_labels, y_pred_full, 'Full VGG: test set')
test_loss_full, test_accuracy_full = full_vgg.evaluate(test64, verbose=0)
print(f'Full VGG test loss: {test_loss_full:.4f}')
print(f'Full VGG test accuracy: {test_accuracy_full:.2%}')
print('The listed off-diagonal counts identify the largest measured class confusions.')
print('Resizing increases image dimensions but does not add new visual information.')
print('This model trains for 10 epochs versus 50, so depth alone cannot explain differences.')

# Understanding deep networks

*   What is the use of activation functions in network? Why is it needed?
*   We have used softmax activation function in the exercise. There are other activation functions available too. What is the difference between sigmoid activation and softmax activation?
*   What is the difference between categorical crossentropy and binary crossentropy loss?

1. **Why activation functions are needed:** Nonlinear activations let a network learn complex relationships. Without them, stacked linear layers collapse into a single linear transformation. ReLU, `max(0, x)`, provides a simple nonlinearity for hidden layers.

2. **Sigmoid versus softmax:** Sigmoid independently maps each logit to a value between 0 and 1. It is suitable for binary classification or multilabel tasks where several labels can be true; its outputs need not sum to one. Softmax converts a vector of logits into probabilities that sum to one, making it suitable for the ten mutually exclusive CIFAR-10 classes. Hidden layers use ReLU, but the output uses softmax to match categorical crossentropy.

3. **Categorical versus binary crossentropy:** Categorical crossentropy is `-sum(y_k * log(p_k))` for mutually exclusive classes, using one-hot targets here. Binary crossentropy is `-[y * log(p) + (1-y) * log(1-p)]` for a binary target and can be applied independently across multilabel outputs. Sparse categorical crossentropy represents the same multiclass objective with integer labels instead of one-hot vectors.